In [ ]:
import random

import numpy
import pandas
from scipy.stats import pointbiserialr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define random seeds for reproducibility.
SEED: int = 42
random.seed(SEED)
numpy.random.seed(SEED)

## **DATASET-COLLEGE-SCORECARD**

[College Scorecard](https://collegescorecard.ed.gov/data/) is the U.S. Department of Education's
institution-level dataset on US colleges and universities: admissions, cost, student demographics,
financial aid, and outcomes (including completion rates), assembled from IPEDS, NSLDS and
SSA/IRS-linked administrative records. It is public domain (US government work), updated
periodically; this notebook uses the **"Most Recent Cohorts (Institution)"** file, downloaded
2026-09-25 from `https://ed-public-download.scorecard.network/downloads/Most-Recent-Cohorts-Institution_06102026.zip`
(dated 2026-06-10, 23 MB zipped / 100 MB CSV, 6,273 rows, 3,308 raw columns). This is the
**cross-sectional** file: **one row per institution** (its most recently reported values), not the
full year-by-year panel. That is a deliberate choice -- see "Why this file, not the full historical
panel" below.

**Why this dataset, for MORSE specifically.** [TableShift](https://tableshift.org/) (Gardner et al.,
NeurIPS 2023 D&B) turned College Scorecard into a distribution-shift benchmark task,
`college_scorecard`: predict `C150_4` (whether at least half of a four-year institution's
first-time, full-time students complete within 150% of normal time) from 119 institutional
features, with a **natural, real (not synthetic) covariate/domain shift**: institutions are split
by their Carnegie Classification "Basic" type (`CCBASIC`), holding out 8 whole institution types
(seminaries, art/design schools, mid-size master's-granting universities, some private
associate's-dominant colleges, ...) as out-of-domain. That is exactly the kind of test MORSE's
robustness claims need and neither of the other two datasets in this repository can provide on
their own: arrhythmia and readmit only have *synthetic* stress (Gaussian noise, prevalence-
preserving corruption, the PC1 population tilt in `evaluation_utils.py`) -- College Scorecard adds
a held-out set of *real* institutions of *genuinely different types* that the model never saw
during selection or training, evaluated with the exact same fitted models.

This notebook reproduces TableShift's feature list and domain-split definition exactly (traced to
their source code, see below, and verified against this data release), but -- deliberately --
**not** their own preprocessing (`categorical_features="label_encode"`, `numeric_features="kbins"`
with 100 bins per column): that scheme is built for gradient-boosted trees, which do not care
whether "80" means fewer or more than "20". MORSE fits an L2 logistic regression and measures the
*sign* of its coefficients, so kbins-encoding away every continuous feature's ordering would
silently defeat the sign-consistency objective. This notebook instead follows the same
leakage-aware preprocessing precedents already set in this repository: fitted statistics (medians,
here) computed on the training split only, the same discipline the arrhythmia imputation-leakage
fix established; an explicit `:missing` indicator for informatively-missing columns, the pattern
used for RadFusion's lab values; and one-hot encoding of nominal categorical codes rather than
leaving them as raw integers, the fix already applied to readmit's `admission_source_id` and
friends -- `region`, `LOCALE`, `CONTROL`, `HIGHDEG`, `CCSIZSET`, `sch_deg`, `STABBR` and
`AccredAgency` are all nominal codes here and are one-hot encoded for the same reason.

**Why this file, not the full historical panel.** TableShift's own 124,699-row task is built from
the *full* multi-year raw archive (`College_Scorecard_Raw_Data_*.zip`, ~470 MB): one row per
institution *per year*, 1996-97 through the present, i.e. roughly 6,000 institutions repeated
~20 times each. MORSE's design (see `README.md`, "Numerical reproducibility") relies on a single
*fixed* train/test split with no institution ever appearing in more than one split -- exactly the
leakage concern already fixed for readmit ("one encounter per patient"). Reproducing TableShift's
full panel would require carrying an institution-id column through the whole pipeline purely to
enforce a group-split, and would inflate the row count with near-duplicate copies of the same
~6,000 institutions across decades without adding independent information. The single-year
cross-section sidesteps this entirely: `UNITID` is unique by construction (verified below), so a
plain stratified split is already leakage-free -- no group-split machinery needed.

In [ ]:
# College Scorecard "Most Recent Cohorts (Institution)", downloaded 2026-09-25.
# https://collegescorecard.ed.gov/data/
raw_header: list[str] = pandas.read_csv("Most-Recent-Cohorts-Institution.csv", nrows=0).columns.tolist()
print(f"raw file: {len(raw_header)} columns")

### The exact TableShift feature list

The target and the 119 predictors below are copied **verbatim** (names only, not TableShift's own
encoding/dtype choices) from
[`tableshift/datasets/college_scorecard.py`](https://github.com/mlfoundations/tableshift/blob/main/tableshift/datasets/college_scorecard.py)
(`COLLEGE_SCORECARD_FEATURES`, fetched 2026-09-25) so that the feature set used here is traceable
to a public, citable definition rather than an ad hoc column pick. Two things do **not** carry over
unmodified, both discovered by diffing this against the actual downloaded file:

- **Column name casing differs** (e.g. TableShift's `AccredAgency` vs. this file's `ACCREDAGENCY`,
  `region` vs. `REGION`, `locale2` vs. `LOCALE2`). Matched case-insensitively below; all 120 names
  (target + 119 predictors) were found this way -- verified, not assumed.
- **`sch_deg`** ("Predominant degree awarded (recoded 0s and 4s)") is a *different raw column* from
  the also-present `PREDDEG` (only 74.7% agreement between them; `sch_deg`'s own values are only
  ever 1/2/3, matching TableShift's "recoded" description) -- see the dedicated note near the
  categorical-encoding cell below.

In [ ]:
TARGET: str = "C150_4"

# Predictors whose raw values can be the literal privacy-suppression sentinel (see the note below
# on which token the CURRENT file actually uses -- it is not the one TableShift's code expects).
SUPPRESSIBLE: list[str] = [
    "loan_ever", "pell_ever", "age_entry", "age_entry_sq", "agege24", "female", "married",
    "dependent", "veteran", "first_gen", "faminc", "md_faminc", "pct_white", "pct_black",
    "pct_asian", "pct_hispanic", "pct_ba", "pct_grad_prof", "pct_born_us", "median_hh_inc",
    "poverty_rate", "unemp_rate",
]

# The 119 predictors of tableshift.datasets.college_scorecard.COLLEGE_SCORECARD_FEATURES, verbatim.
PREDICTORS: list[str] = [
    "STABBR", "AccredAgency", "sch_deg", "main", "NUMBRANCH", "HIGHDEG", "CONTROL", "region",
    "LOCALE", "locale2", "CCBASIC", "CCSIZSET", "HBCU", "ADM_RATE", "ADM_RATE_ALL", "SATVRMID",
    "SATMTMID", "SATWRMID", "ACTCMMID", "ACTENMID", "ACTMTMID", "ACTWRMID",
    "PCIP01", "PCIP03", "PCIP04", "PCIP05", "PCIP09", "PCIP10", "PCIP11", "PCIP12", "PCIP13",
    "PCIP14", "PCIP15", "PCIP16", "PCIP19", "PCIP22", "PCIP23", "PCIP24", "PCIP25", "PCIP26",
    "PCIP27", "PCIP29", "PCIP30", "PCIP31", "PCIP38", "PCIP39", "PCIP40", "PCIP41", "PCIP42",
    "PCIP43", "PCIP44", "PCIP45", "PCIP46", "PCIP47", "PCIP48", "PCIP49", "PCIP50", "PCIP51",
    "PCIP52", "PCIP54",
    "DISTANCEONLY", "UGDS", "UG", "UGDS_WHITE", "UGDS_BLACK", "UGDS_HISP", "UGDS_ASIAN",
    "UGDS_AIAN", "UGDS_NHPI", "UGDS_2MOR", "UGDS_NRA", "UGDS_UNKN", "UGDS_WHITENH",
    "UGDS_BLACKNH", "UGDS_API", "UGDS_AIANOld", "UGDS_HISPOld", "UG_NRA", "UG_UNKN",
    "UG_WHITENH", "UG_BLACKNH", "UG_API", "UG_AIANOld", "UG_HISPOld",
    "PPTUG_EF", "PPTUG_EF2", "NPT4_PROG", "COSTT4_A", "COSTT4_P", "TUITIONFEE_IN",
    "TUITIONFEE_OUT", "TUITIONFEE_PROG", "TUITFTE", "INEXPFTE", "AVGFACSAL", "PFTFAC", "PCTPELL",
] + SUPPRESSIBLE
assert len(PREDICTORS) == 119, len(PREDICTORS)

# Case-insensitive match against the raw header (see the note above on casing).
upper_to_raw: dict[str, str] = {h.upper(): h for h in raw_header}
_missing_from_raw: list[str] = [c for c in [TARGET] + PREDICTORS if c.upper() not in upper_to_raw]
assert not _missing_from_raw, f"not found in the raw file: {_missing_from_raw}"
print(f"all {1 + len(PREDICTORS)} TableShift columns (target + predictors) found in the raw file")

### Loading, and a genuine schema-drift finding: the privacy-suppression sentinel is now `"PS"`

`SUPPRESSIBLE` are the columns TableShift's own source marks
`na_values=('PrivacySuppressed',)` -- values withheld by the Department of Education when too few
students would make the underlying figure individually identifiable. **Checked directly against
this file: the literal string `"PrivacySuppressed"` never appears in it.** Scanning every
non-numeric token actually present in these 21 columns finds exactly one: **`"PS"`**. This is a
genuine difference between the 2026-06-10 data release and whatever vintage TableShift's code
comment was written against (College Scorecard has changed this convention at least once) --
not a guess: verified by listing every non-numeric token that occurs and finding only `"PS"`, on
this exact file. Using `"PrivacySuppressed"` here (i.e. trusting the old comment literally) would
silently leave every suppressed cell as the string `"PS"`, which then fails to parse as a number at
all downstream. `na_values` below is set to `"PS"` accordingly, and every affected column is then
coerced to numeric explicitly (`errors="raise"`, so a *third*, still-different sentinel in some
future release would fail loudly here rather than silently produce garbage).

In [ ]:
usecols: list[str] = [upper_to_raw[c.upper()] for c in [TARGET] + PREDICTORS] + \
    [upper_to_raw[c.upper()] for c in ("UNITID", "INSTNM", "CURROPER")]
na_values: dict[str, list[str]] = {upper_to_raw[c.upper()]: ["PS"] for c in SUPPRESSIBLE}

original_data: pandas.DataFrame = pandas.read_csv(
    "Most-Recent-Cohorts-Institution.csv", usecols=usecols, na_values=na_values, low_memory=False)
original_data = original_data.rename(
    columns={upper_to_raw[c.upper()]: c for c in [TARGET] + PREDICTORS + ["UNITID", "INSTNM", "CURROPER"]})

for column in SUPPRESSIBLE:
    original_data[column] = pandas.to_numeric(original_data[column], errors="raise")

print(f"loaded: {original_data.shape[0]} institutions, {original_data.shape[1]} columns "
      f"({len(PREDICTORS)} predictors + target + idx/name/status)")
print("UNITID uniquely identifies a row (one row per institution):", original_data["UNITID"].is_unique)

### Row filters

Three filters are applied, each printed with its exact effect on the row count, in this order:

1. **`CURROPER == 1` (currently operating).** A closed institution's most-recent snapshot is
   frozen data from whenever it stopped reporting -- not a live case, and a fair number of closures
   are for-cause (financial collapse, loss of accreditation) rather than a representative "just
   like the open ones" sample.
2. **`C150_4` present.** This completion-rate measure is only defined for institutions that track a
   first-time, full-time bachelor's-seeking cohort (IPEDS' Graduation Rate Survey scope) -- most
   institutions without it are certificate- or associate's-dominant, so this filter is doing double
   duty as "select the institutions the target actually applies to", not just "drop rows with a
   missing label". It is *correlated with, but not strictly gated by,* `PREDDEG` (the predominant-
   credential-level code): of the institutions that survive this filter (and the `CCBASIC` filter
   below), 1,751 are `PREDDEG == 3` (predominantly bachelor's), but 310 are `PREDDEG == 2`
   (predominantly associate's) and 145 are `PREDDEG == 1` (predominantly certificate) -- presumably
   institutions that also run a smaller bachelor's-track cohort alongside their main programs.
3. **`CCBASIC` classified** (drop `-2` "not applicable" and `0` "not classified"). `CCBASIC` is the
   variable the covariate shift is built on (next section); a handful of institutions have no
   Carnegie classification on file at all and cannot be assigned to either side of that split.

In [ ]:
_n0: int = len(original_data)
original_data = original_data[original_data["CURROPER"] == 1].copy()
print(f"[filter 1] currently operating: {_n0} -> {len(original_data)} "
      f"({_n0 - len(original_data)} closed institutions dropped)")

_n1: int = len(original_data)
original_data = original_data[original_data[TARGET].notna()].copy()
print(f"[filter 2] {TARGET} present: {_n1} -> {len(original_data)} "
      f"({_n1 - len(original_data)} dropped -- not primarily-bachelor's/4-year, or no trackable cohort)")

original_data["CCBASIC"] = original_data["CCBASIC"].astype("Int64")
_n2: int = len(original_data)
original_data = original_data[original_data["CCBASIC"].notna() & ~original_data["CCBASIC"].isin([-2, 0])].copy()
print(f"[filter 3] CCBASIC classified: {_n2} -> {len(original_data)} ({_n2 - len(original_data)} dropped)")

assert original_data["UNITID"].is_unique
print(f"\nfinal candidate pool: {len(original_data)} institutions")

### Target

`label = 1[C150_4 > 0.5]`, matching TableShift's own `preprocess_college_scorecard` exactly (so
that a reported AUC here means the same prediction task as TableShift's own benchmark numbers).
`C150_4` is a completion-rate *fraction* in `[0, 1]` (verified: `min=0.0`, `max=1.0` on this data),
so `> 0.5` reads as "did at least half of this institution's tracked cohort finish within 150% of
normal time".

In [ ]:
original_data["label"] = (original_data[TARGET] > 0.5).astype(int)
print(f"target prevalence, whole candidate pool: {original_data['label'].mean():.3f} "
      f"({int(original_data['label'].sum())} / {len(original_data)})")

### The covariate shift: `CCBASIC`, and a second schema-drift finding

TableShift's `college_scorecard` experiment config
([`tableshift/configs/benchmark_configs.py`](https://github.com/mlfoundations/tableshift/blob/main/tableshift/configs/benchmark_configs.py))
splits institutions by `CCBASIC` (Carnegie Classification -- Basic), holding out 8 whole
categories as out-of-domain, given there as **text labels**:

```
Special Focus Institutions--Other special-focus institutions
Special Focus Institutions--Theological seminaries, Bible colleges, and other faith-related institutions
Associate's--Private For-profit 4-year Primarily Associate's
Baccalaureate Colleges--Diverse Fields
Special Focus Institutions--Schools of art, music, and design
Associate's--Private Not-for-profit
Baccalaureate/Associate's Colleges
Master's Colleges and Universities (larger programs)
```

**In the current file, `CCBASIC` is a numeric code (1-33, plus the `-2`/`0` sentinels already
filtered above), not text** -- another schema-drift instance, the same kind as the `"PS"` sentinel
above. The code-to-label mapping is **not** in the current
[data dictionary](https://collegescorecard.ed.gov/assets/CollegeScorecardDataDictionary.xlsx)
(it lists only the `-2` sentinel for `CCBASIC`; the label text was dropped from that file at some
point). It **is** still published by IPEDS itself, as the `CCBASIC` variable of the "Carnegie
Classification 2005/2010: Basic" survey
(`nces.ed.gov`, `IPEDS<year>Tablesdoc.xlsx`, sheet `valuesets21`) -- confirmed to be the *exact*
right table three ways: (1) the variable name matches (`CCBASIC`, not the newer `C15BASIC` /
`C18BASIC` / `C21BASIC`, which use a different, incompatible code set for the same categories),
(2) every one of TableShift's 8 label strings matches an entry in this table character-for-character
once TableShift's own `"Special Focus Institutions--"` prefix is normalised to the source table's
`"Special Focus--"`, and (3) spot-checking institution names by code confirms it directly: code 24
(mapped to "faith-related institutions") contains Moody Bible Institute and several rabbinical
colleges/yeshivas; code 15 ("Research Universities, very high research activity", **not** held out)
contains MIT and Caltech; code 18 ("Master's, larger programs", held out) contains Youngstown State
and Kutztown University -- exactly as expected. The 8 held-out codes, asserted below to match
TableShift's 8 strings exactly, are **`{9, 14, 18, 22, 23, 24, 30, 32}`**.

**What this buys MORSE, empirically (verified below with a plain all-features L2 logistic
regression, not yet MORSE's own selection):** the in-domain (ID) and out-of-domain (OOD) sets
differ not just in composition but in the *label prior* too (a real, external analogue of the
`radfusion` train/val/test prevalence shift, and of TableShift's own published finding that most of
the ID-to-OOD benchmark gap is driven by label-distribution shift, not by the features changing
meaning) -- and a plain all-features model's ROC-AUC drops substantially from ID-test to OOD-test.
This is a *genuine* stress test standing next to MORSE's two *synthetic* ones (the Gaussian /
prevalence-preserving noise sweeps, and the PC1 population tilt), evaluated on the exact same fitted
models -- see the diagnostic cell at the end of this notebook.

In [ ]:
# CCBASIC code -> label (IPEDS "Carnegie Classification 2005/2010: Basic"; see the note above).
CCBASIC_LABELS: dict[int, str] = {
    1: "Associate's--Public Rural-serving Small", 2: "Associate's--Public Rural-serving Medium",
    3: "Associate's--Public Rural-serving Large", 4: "Associate's--Public Suburban-serving Single Campus",
    5: "Associate's--Public Suburban-serving Multicampus", 6: "Associate's--Public Urban-serving Single Campus",
    7: "Associate's--Public Urban-serving Multicampus", 8: "Associate's--Public Special Use",
    9: "Associate's--Private Not-for-profit", 10: "Associate's--Private For-profit",
    11: "Associate's--Public 2-year colleges under 4-year universities",
    12: "Associate's--Public 4-year Primarily Associate's",
    13: "Associate's--Private Not-for-profit 4-year Primarily Associate's",
    14: "Associate's--Private For-profit 4-year Primarily Associate's",
    15: "Research Universities (very high research activity)",
    16: "Research Universities (high research activity)", 17: "Doctoral/Research Universities",
    18: "Master's Colleges and Universities (larger programs)",
    19: "Master's Colleges and Universities (medium programs)",
    20: "Master's Colleges and Universities (smaller programs)",
    21: "Baccalaureate Colleges--Arts & Sciences", 22: "Baccalaureate Colleges--Diverse Fields",
    23: "Baccalaureate/Associate's Colleges",
    24: "Special Focus--Theological seminaries, Bible colleges, and other faith-related institutions",
    25: "Special Focus--Medical schools and medical centers",
    26: "Special Focus--Other health professions schools", 27: "Special Focus--Schools of engineering",
    28: "Special Focus--Other technology-related schools",
    29: "Special Focus--Schools of business and management",
    30: "Special Focus--Schools of art, music, and design", 31: "Special Focus--Schools of law",
    32: "Special Focus--Other special-focus institutions", 33: "Special Focus--Tribal Colleges",
}

# TableShift's 8 held-out labels, verbatim (see the note above for the source and the prefix
# normalisation used to match them against CCBASIC_LABELS).
_TS_OOD_LABELS: list[str] = [
    "Special Focus Institutions--Other special-focus institutions",
    "Special Focus Institutions--Theological seminaries, Bible colleges, and other faith-related institutions",
    "Associate's--Private For-profit 4-year Primarily Associate's",
    "Baccalaureate Colleges--Diverse Fields",
    "Special Focus Institutions--Schools of art, music, and design",
    "Associate's--Private Not-for-profit",
    "Baccalaureate/Associate's Colleges",
    "Master's Colleges and Universities (larger programs)",
]
_normalise = lambda s: s.replace("Special Focus Institutions--", "Special Focus--")
OOD_CCBASIC_CODES: set[int] = {code for code, label in CCBASIC_LABELS.items()
                               if _normalise(label) in [_normalise(x) for x in _TS_OOD_LABELS]}
assert len(OOD_CCBASIC_CODES) == 8 == len(_TS_OOD_LABELS), OOD_CCBASIC_CODES
print("OOD CCBASIC codes (verified by exact label match against TableShift's 8 strings):")
for _code in sorted(OOD_CCBASIC_CODES):
    print(f"  {_code:2d}  {CCBASIC_LABELS[_code]}")

original_data["is_ood"] = original_data["CCBASIC"].astype(int).isin(OOD_CCBASIC_CODES)
print(f"\nID (in-domain):  {(~original_data['is_ood']).sum():4d} institutions, "
      f"target prevalence {original_data.loc[~original_data['is_ood'], 'label'].mean():.3f}")
print(f"OOD (held-out):  {original_data['is_ood'].sum():4d} institutions, "
      f"target prevalence {original_data.loc[original_data['is_ood'], 'label'].mean():.3f}")

### Columns that carry no signal in this data vintage

`CCBASIC` is used only for the split above and is now removed from the feature set -- keeping it
would let the model detect which domain a row came from directly, rather than being tested on
whether it generalises across domains. Beyond that, some of TableShift's 119 predictors are
**entirely or almost entirely empty** in the 2026-06-10 release (columns College Scorecard has
since stopped populating -- e.g. several non-Hispanic / alternate race-share breakdowns, and the
whole `UG*` "all undergraduates" block, superseded by the `UGDS*` "degree-seeking" block that
*is* still populated). Two thresholds, both checked and printed below rather than assumed:

- **100% missing -> dropped.** A constant-NaN column carries no information and, once imputed, would
  be a constant column -- pure padding for the GA's search space.
- **>85% missing (but not 100%) -> dropped.** `COSTT4_P`/`TUITIONFEE_PROG` (program-year-institution
  costs; our filtered population is almost entirely academic-year institutions) and `ACTWRMID` (the
  ACT Writing section, largely discontinued) leave only a handful of real, non-imputed values --
  too sparse to carry a reliable train-median imputation on top of everything else, and the
  institution-type information they gate on is already captured elsewhere (`CONTROL`, `HIGHDEG`).

In [ ]:
feature_cols: list[str] = [c for c in PREDICTORS if c != "CCBASIC"]
missingness: pandas.Series = original_data[feature_cols].isna().mean().sort_values(ascending=False)

DROP_ALWAYS_MISSING: list[str] = missingness[missingness == 1.0].index.tolist()
DROP_NEAR_MISSING: list[str] = missingness[(missingness > 0.85) & (missingness < 1.0)].index.tolist()

print(f"dropping {len(DROP_ALWAYS_MISSING)} columns that are 100% missing:")
print(" ", DROP_ALWAYS_MISSING)
print(f"\ndropping {len(DROP_NEAR_MISSING)} columns that are >85% missing:")
for _c in DROP_NEAR_MISSING:
    print(f"    {_c}: {missingness[_c]:.1%} missing")

feature_cols = [c for c in feature_cols if c not in DROP_ALWAYS_MISSING + DROP_NEAR_MISSING]
print(f"\nremaining predictor columns: {len(feature_cols)}")

### Train / ID-test / OOD-test split

The split happens **before** any imputation statistic is computed, so that every train-only
statistic used below (medians, one-hot's TRAIN-only cleanup) is fit on the training rows alone --
the same leakage discipline already applied to arrhythmia and readmit. Only the **ID** pool is
split further (stratified 70/30 on the label, fixed `SEED`, matching the "single fixed split" design
the rest of this repository relies on -- see `README.md`, "Numerical reproducibility"); the whole
**OOD** pool is held out as a third, untouched file. Three CSVs come out of this notebook:

- `college_scorecard_preprocessed_train_data.csv` -- what `CSV_TRAIN_PATH` in
  `training_notebook.ipynb` should point to,
- `college_scorecard_preprocessed_test_data.csv` -- `CSV_TEST_PATH`, an in-domain held-out set,
  same role as arrhythmia's / readmit's test file,
- `college_scorecard_preprocessed_ood_test_data.csv` -- **not** part of the standard MORSE
  pipeline (`training_notebook.ipynb` only loads two files); a genuine external-shift set for a
  supplemental evaluation, evaluating the models MORSE already trained/selected on the ID data,
  exactly the way `evaluation_utils.evaluate_model` already works on any (model, X, y) triple.

In [ ]:
id_data: pandas.DataFrame = original_data[~original_data["is_ood"]].copy()
ood_data: pandas.DataFrame = original_data[original_data["is_ood"]].copy()

train_idx, test_idx = train_test_split(
    id_data.index, test_size=0.3, stratify=id_data["label"], random_state=SEED)
train_data: pandas.DataFrame = id_data.loc[train_idx].copy()
test_data: pandas.DataFrame = id_data.loc[test_idx].copy()

print(f"train:    {len(train_data):4d} institutions, {train_data['label'].mean():.3f} positive")
print(f"id_test:  {len(test_data):4d} institutions, {test_data['label'].mean():.3f} positive")
print(f"ood_test: {len(ood_data):4d} institutions, {ood_data['label'].mean():.3f} positive")

### Missing-value handling for continuous predictors

For every continuous predictor that has *any* missing values, an explicit `<column>:missing`
indicator is added (1 = this institution did not report the value) and the value itself is filled
with the **training split's own median** -- the same "keep the flag, impute the value from train
only" pattern already used for RadFusion's lab values, for the same reason: a lab test not being
ordered, or an institution not reporting its SAT scores, is itself informative (non-selective or
test-optional institutions are exactly the ones that do not report a school-wide SAT midpoint), so
folding the flag into the value would mix two different effects into one coefficient. One
consistent rule is used -- *any* missingness gets a flag, no separate threshold to argue about --
rather than picking a cutoff by hand.

In [ ]:
ALREADY_BINARY: set[str] = {"main", "HBCU", "DISTANCEONLY"}   # raw 0/1 flags, no encoding needed
CATEGORICAL: set[str] = {"STABBR", "AccredAgency", "sch_deg", "HIGHDEG", "CONTROL", "region",
                         "LOCALE", "CCSIZSET"}                # nominal codes -> one-hot below

categorical_cols: list[str] = [c for c in feature_cols if c in CATEGORICAL]
binary_cols: list[str] = [c for c in feature_cols if c in ALREADY_BINARY]
continuous_cols: list[str] = [c for c in feature_cols if c not in CATEGORICAL and c not in ALREADY_BINARY]
assert set(categorical_cols) | set(binary_cols) | set(continuous_cols) == set(feature_cols)

cols_with_missing: list[str] = [c for c in continuous_cols if original_data[c].isna().any()]
print(f"{len(cols_with_missing)} of {len(continuous_cols)} continuous columns get a "
      f"`:missing` indicator + train-median imputation")

for _frame in (train_data, test_data, ood_data):
    for _c in cols_with_missing:
        _frame[f"{_c}:missing"] = _frame[_c].isna().astype(int)

train_medians: pandas.Series = train_data[continuous_cols].median()
assert train_medians.notna().all(), train_medians[train_medians.isna()]
for _frame in (train_data, test_data, ood_data):
    for _c in continuous_cols:
        _frame[_c] = _frame[_c].fillna(train_medians[_c])

print("\nno remaining NaN in the continuous columns of any split:",
      all(_frame[continuous_cols].isna().sum().sum() == 0 for _frame in (train_data, test_data, ood_data)))

### Categorical encoding

Every categorical column above is a **nominal** code (state, accreditor, Carnegie size/setting,
locale, control type, ...), never ordinal, and is one-hot encoded accordingly (`drop_first=True`,
matching readmit's convention) -- leaving any of these as a raw integer, the way `admission_type_id`
/ `discharge_disposition_id` / `admission_source_id` were originally left in the readmit data before
that was fixed, would hand the model a meaningless numeric ordering and produce meaningless
coefficients and sign-consistency checks.

`HIGHDEG` (0-4) is mapped to TableShift's own human-readable labels
(`{0: "Non-degree-granting", ..., 4: "Graduate degree"}`) so the resulting one-hot columns are
self-explanatory. **`sch_deg` gets its *own*, different label set, not `HIGHDEG`'s**: despite using
overlapping integer codes, `sch_deg` (the Title-IV "predominant degree" classification) is a
different raw column from `PREDDEG`/`HIGHDEG` -- verified above, only 74.7% agreement -- and its
values are only ever 1/2/3 in this data (matching TableShift's "recoded 0s and 4s" description,
already done upstream by College Scorecard: `0` folded into one of 1-3, `4` folded into `3`), so it
is labelled `{1: "Certificate", 2: "Associate's", 3: "Bachelor's or higher"}`.

Categories are one-hot encoded on the **combined** train+test+ood_test data (so all three CSVs end
up with identical columns in the identical order -- `training_notebook.ipynb` loads train and test
as separate files and needs their columns to line up) *before* the split-specific cleanup below --
the same convention readmit's own `pandas.get_dummies` call already uses: which category labels
exist is shared across splits, but (per the cell above) no continuous statistic and no label ever
is. (Arrhythmia has no categorical columns to encode -- its UCI source is already an all-numeric
feature matrix -- so it sets no precedent either way here.)

In [ ]:
HIGHDEG_LABELS: dict[int, str] = {0: "Non-degree-granting", 1: "Certificate degree",
                                  2: "Associate degree", 3: "Bachelor's degree", 4: "Graduate degree"}
SCH_DEG_LABELS: dict[int, str] = {1: "Certificate", 2: "Associate's", 3: "Bachelor's or higher"}

combined: pandas.DataFrame = pandas.concat(
    [train_data, test_data, ood_data], keys=["train", "test", "ood"])

for _col, _labels in (("HIGHDEG", HIGHDEG_LABELS), ("sch_deg", SCH_DEG_LABELS)):
    combined[_col] = combined[_col].map(lambda v: _labels.get(int(v)) if pandas.notna(v) else v)
for _col in categorical_cols:
    combined[_col] = combined[_col].astype("object").where(combined[_col].notna(), "Missing").astype(str)

combined = pandas.get_dummies(combined, columns=categorical_cols, drop_first=True)
train_data = combined.xs("train")
test_data = combined.xs("test")
ood_data = combined.xs("ood")

onehot_cols: list[str] = [c for c in combined.columns
                          if any(c.startswith(f"{cat}_") for cat in categorical_cols)]
final_feature_cols: list[str] = (binary_cols + continuous_cols
                                 + [f"{c}:missing" for c in cols_with_missing] + onehot_cols)
print(f"feature count before the train-only cleanup below: {len(final_feature_cols)} "
      f"({len(binary_cols)} already-binary + {len(continuous_cols)} continuous + "
      f"{len(cols_with_missing)} missing-flags + {len(onehot_cols)} one-hot levels)")

### Train-only cleanup: zero-variance and exact-duplicate columns

Two problems only show up once the categoricals are one-hot encoded and the data is actually
split, checked here **against the training split alone** (test/OOD are never consulted to decide
what to drop -- only to confirm the result is sane):

**(a) Zero-variance-in-train columns.** A one-hot level for a rare category -- a handful of
institutions nationally, e.g. `STABBR` levels for small US territories, or a niche accreditor --
can land entirely in the test or OOD split by chance of the split, leaving an all-zero column in
train. Such a column cannot receive a meaningful coefficient (a constant input has an arbitrary
fitted slope under L2 regularisation) and its marginal correlation with the label is *exactly* zero
by construction (`evaluation_utils.compute_marginal_correlations` returns 0 for any
constant column) -- so if the GA ever selects it, it would be automatically flagged
sign-inconsistent for a reason that has nothing to do with MORSE's actual hypothesis. Dropped.

**(b) Exact-duplicate columns.** Several `:missing` indicators turn out to be **byte-identical to
each other on every training row** -- e.g. `SATVRMID:missing == SATMTMID:missing` (the SAT
verbal and math sections are reported together or not at all) and a cluster of 7 Census-sourced
columns (`agege24`, the four `pct_*` race shares, `median_hh_inc`, `poverty_rate`, `unemp_rate`)
that all come from one zip-code-linkage step which succeeds or fails for an institution as a single
unit. This is exactly the same kind of deterministic redundancy as the RadFusion ICD
`:presence`/`:frequency` columns: keeping every copy adds collinearity without adding information,
and collinear duplicates are precisely the structure that produces sign-inconsistent coefficients.
Only the first column of each identical group is kept; every group found is printed below so the
merge is fully visible, not a silent drop.

In [ ]:
train_features: pandas.DataFrame = train_data[final_feature_cols]

nunique_train: pandas.Series = train_features.nunique()
zero_variance: list[str] = nunique_train[nunique_train <= 1].index.tolist()
print(f"[cleanup a] zero-variance-in-train columns dropped: {len(zero_variance)}")
print(" ", zero_variance)

kept_after_a: list[str] = [c for c in final_feature_cols if c not in zero_variance]
_train_arr = train_data[kept_after_a].to_numpy()
_signature_of: dict[bytes, str] = {}
duplicate_groups: dict[str, list[str]] = {}
for _i, _c in enumerate(kept_after_a):
    _sig = _train_arr[:, _i].tobytes()
    if _sig in _signature_of:
        duplicate_groups.setdefault(_signature_of[_sig], []).append(_c)
    else:
        _signature_of[_sig] = _c

dropped_duplicates: list[str] = [c for group in duplicate_groups.values() for c in group]
print(f"\n[cleanup b] exact-duplicate-in-train columns dropped: {len(dropped_duplicates)} "
      f"({len(duplicate_groups)} groups)")
for _keeper, _dupes in duplicate_groups.items():
    print(f"    kept {_keeper!r}  <-  identical (on every train row) to {_dupes}")

final_feature_cols = [c for c in kept_after_a if c not in dropped_duplicates]

_nun_train = train_data[final_feature_cols].nunique()
assert (_nun_train > 1).all(), _nun_train[_nun_train <= 1].index.tolist()
print(f"\nfinal feature count after cleanup: {len(final_feature_cols)}")
print("verified: no zero-variance or exact-duplicate column remains in TRAIN")

### Save

`idx`-style identifiers (`UNITID`, `INSTNM`) were used only to verify uniqueness above and are
**not** written to the output files -- College Scorecard is a single flat table with no other
modality to link back to (unlike RadFusion's `idx`, kept deliberately for that reason), so there is
nothing to gain from carrying an identifier into the model input, matching arrhythmia's and
readmit's convention of not persisting one either.

In [ ]:
OUTPUT_COLUMNS: list[str] = ["label"] + final_feature_cols
for _name, _frame in (("train", train_data), ("test", test_data), ("ood_test", ood_data)):
    assert _frame[OUTPUT_COLUMNS].isna().sum().sum() == 0, f"{_name}: unexpected NaN before saving"
    _out_path = f"college_scorecard_preprocessed_{_name}_data.csv"
    _frame[OUTPUT_COLUMNS].to_csv(_out_path, index=False)
    print(f"saved {_out_path}: {_frame.shape[0]} rows, {len(OUTPUT_COLUMNS)} columns "
          f"({_frame['label'].mean():.3f} positive)")

## Diagnostic: is this a good MORSE test-bed?

Two checks, using a single plain all-features L2 logistic regression fit on the training split
(**not** MORSE's own feature selection -- this is a property of the *dataset*, computed the same
way it was for arrhythmia, so the two are directly comparable):

1. **Sign-flip screening statistic** -- the fraction of the fitted coefficients whose sign
   disagrees with their feature's own marginal correlation with the label (the exact quantity
   `evaluation_utils.compute_model_sign_consistency` measures, `1 - sign_consistency`). This is the
   structural property MORSE's second objective is designed to fix; a dataset with (close to) 0%
   sign-flips gives the sign-consistency objective nothing to do.
2. **The ID -> OOD AUC gap** -- how much a model that has never seen the 8 held-out Carnegie
   classes degrades on them, the real-world stress test this dataset adds on top of MORSE's two
   synthetic ones.

In [ ]:
X_train: pandas.DataFrame = train_data[final_feature_cols]
y_train: numpy.ndarray = train_data["label"].to_numpy()
X_test: pandas.DataFrame = test_data[final_feature_cols]
y_test: numpy.ndarray = test_data["label"].to_numpy()
X_ood: pandas.DataFrame = ood_data[final_feature_cols]
y_ood: numpy.ndarray = ood_data["label"].to_numpy()

scaler: StandardScaler = StandardScaler()
X_train_scaled: numpy.ndarray = scaler.fit_transform(X_train.astype(float))
X_test_scaled: numpy.ndarray = scaler.transform(X_test.astype(float))
X_ood_scaled: numpy.ndarray = scaler.transform(X_ood.astype(float))

model: LogisticRegression = LogisticRegression(solver="lbfgs", max_iter=2000, random_state=SEED)
model.fit(X_train_scaled, y_train)

roc_test: float = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
pr_test: float = average_precision_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
roc_ood: float = roc_auc_score(y_ood, model.predict_proba(X_ood_scaled)[:, 1])
pr_ood: float = average_precision_score(y_ood, model.predict_proba(X_ood_scaled)[:, 1])
print(f"all-features L2-LR  |  id_test:  ROC-AUC={roc_test:.4f}  PR-AUC={pr_test:.4f}")
print(f"all-features L2-LR  |  ood_test: ROC-AUC={roc_ood:.4f}  PR-AUC={pr_ood:.4f}")
print(f"ROC-AUC drop, id_test -> ood_test: {roc_test - roc_ood:+.4f}")

# Marginal correlation of every training feature with the training label (Matthews for binary
# columns, point-biserial otherwise -- the same rule evaluation_utils.compute_marginal_correlations
# uses).
marginal_corr: numpy.ndarray = numpy.zeros(len(final_feature_cols))
for _j, _col in enumerate(final_feature_cols):
    _values: numpy.ndarray = X_train[_col].to_numpy(dtype=float)
    _n_unique: int = len(numpy.unique(_values))
    if _n_unique <= 1:
        marginal_corr[_j] = 0.0
    elif _n_unique == 2:
        marginal_corr[_j] = matthews_corrcoef(y_train, _values.astype(int))
    else:
        marginal_corr[_j], _ = pointbiserialr(y_train, _values)

check: numpy.ndarray = marginal_corr * model.coef_[0]
inconsistent: numpy.ndarray = (check < 0) | numpy.isclose(check, 0.0, atol=1e-12)
print(f"\nsign-flip screen, all {len(final_feature_cols)} features: "
      f"{inconsistent.sum()}/{len(final_feature_cols)} inconsistent ({inconsistent.mean():.1%}) "
      f"-- arrhythmia reference: 41.4% (278 features)")

_order: numpy.ndarray = numpy.argsort(-numpy.abs(marginal_corr))
for _k in (5, 15, 40, 100):
    _idx = _order[:_k]
    _m = LogisticRegression(solver="lbfgs", max_iter=2000, random_state=SEED)
    _m.fit(X_train_scaled[:, _idx], y_train)
    _check_k = marginal_corr[_idx] * _m.coef_[0]
    _inc_k = (_check_k < 0) | numpy.isclose(_check_k, 0.0, atol=1e-12)
    print(f"  top-{_k:3d} by |marginal corr|: {_inc_k.sum()}/{_k} inconsistent ({_inc_k.mean():.1%})")